In [0]:
import pyspark.sql.functions as F

In [0]:
RENAME_MAP = {
  'tourney_name' : 'tournament_name',
  'tourney_level' : 'tournament_level',
  'tourney_date' : 'tournament_date',
  'match_num' : 'match_number',
  'minutes' : 'match_duration_minutes'
}
tourney_level_map = {
  "G": "Grand Slam",
  "M": "Masters 1000",
  "A": "ATP",
  "C": "Challenger",
  "F": "Futures",
  "D": "Davis Cup",
  "O": "Olympics",
  "S": "Special",
  "R": "Round-robin",
  "L": "Laver Cup"
}
entry_map = {
  "Q": "Qualifier",
  "WC": "Wildcard",
  "LL": "Lucky Loser",
  "ALT": "Alternate",
  "Alt": "Alternate",
  "AL": "Alternate",
  "PR": "Protected Ranking",
  "SE": "Special Exempt",
  "JE": "Junior Exempt",
  "JR": "Junior Exempt",
  "ITF": "ITF entry",
  "IR": "Injury Replacement",
  "A": "Direct Acceptance"
}
round_map = {
  'R128' : 'Round of 128',
  'R64' : 'Round of 64',
  'R32' : 'Round of 32',
  'R16' : 'Round of 16',
  'QF' : 'Quarter-final',
  'SF' : 'Semi-final',
  'F' : 'Final'
}

#Pulling data from bronze

In [0]:
df = spark.table('tennislakehouse.bronze.matches')

#Cleaning the data

In [0]:
#cleaning column surface
df = df.withColumn(
  'surface',
  F.when(
    ~F.col('surface').isin('Hard', 'Grass', 'Clay', 'Carpet') | F.col('surface').isNull(), 
    'N/A'
  ).otherwise(F.trim(F.col('surface')))
)

In [0]:
#cleaning column draw_size
df = df.withColumn(
  'draw_size',
  F.when(
    ~F.col('draw_size').isin('2', '4', '8', '16', '32', '64', '128'),
    0
  ).otherwise(F.trim(F.col('draw_size'))).cast('int')
)

In [0]:
#cleaning tourney_level
tourney_mapper = F.create_map(
    [F.lit(x) for x in sum(tourney_level_map.items(), ())]
)

df = df.withColumn(
    'tourney_level',
    F.when(
        ~F.col('tourney_level').isin(list(tourney_level_map.keys())) | F.col('tourney_level').isNull(),
        'N/A'
    ).otherwise(tourney_mapper[F.col('tourney_level')])
)

In [0]:
#cleaning tourney_date
df = df.withColumn(
    'tourney_date',
    F.try_to_date((F.col('tourney_date').cast('string')), 'yyyyMMdd')
)

In [0]:
#cleaning match_num
df = df.withColumn(
    'match_num',
    F.when(
        (F.col('match_num').try_cast('int') > 128) |
        (F.col('match_num').try_cast('int').isNull()),
        0
    ).otherwise(F.col('match_num').cast('int'))
)

In [0]:
#cleaning winner_seed and loser_seed
df = df.withColumn(
    'winner_seed',
    F.when(
        (F.col("winner_seed").try_cast('float') < 1) |
        (F.col("winner_seed").try_cast('float') > 32) |
         F.col("winner_seed").isNull() |
         F.col('winner_seed').try_cast('float').isNull(),
        0
    ).otherwise(F.col("winner_seed").cast('float').cast('int'))
)
df = df.withColumn(
    'loser_seed',
    F.when(
        (F.col("loser_seed").try_cast('float') < 1) |
        (F.col("loser_seed").try_cast('float') > 32) |
        F.col("loser_seed").isNull() |
        F.col('loser_seed').try_cast('float').isNull(),
        0
    ).otherwise(F.col("loser_seed").cast('float').cast('int'))
)

In [0]:
#Cleaning winner_entry and loser_entry
entry_mapper = F.create_map(
    [F.lit(x) for x in sum(entry_map.items(), ())]
)

df = df.withColumn(
    'winner_entry',
    F.when(
        ~F.col("winner_entry").isin(list(entry_map.keys())) | F.col('winner_entry').isNull(),
        'N/A'
    ).otherwise(entry_mapper[F.col('winner_entry')])
)
df = df.withColumn(
    'loser_entry',
    F.when(
        ~F.col("loser_entry").isin(list(entry_map.keys())) | F.col('loser_entry').isNull(),
        'N/A'
    ).otherwise(entry_mapper[F.col('loser_entry')])
)

In [0]:
#Cleaning winner_age and loser_age
df = df.withColumn(
    'winner_age',
    F.col("winner_age").cast('int')
)
df = df.withColumn(
    'loser_age',
    F.col("loser_age").cast('int')
)

In [0]:
#cleaning score
df = df.withColumn(
    'score',
    F.when(
        (~F.col('score').rlike(r"^(\d+-\d+)( \d+-\d+)*$")) | (F.col('score').isNull()),
        'N/A'
    ).otherwise(F.trim(F.col('score')))
)

In [0]:
#cleaning best_of
df = df.withColumn(
    'best_of',
    F.when(
        ~F.col("best_of").isin('1', '3', '5') | F.col("best_of").isNull(),
        0
    ).otherwise(F.col("best_of").cast('int'))
)

In [0]:
#cleaning round
round_mapper = F.create_map(
    [F.lit(x) for x in sum(round_map.items(), ())]
)

df = df.withColumn(
    'round',
    F.when(
        ~F.col("round").isin(list(round_map.keys())) | F.col("round").isNull(),
        'N/A'
    ).otherwise(round_mapper[F.col("round")])
)

In [0]:
#cleaning minutes
df = df.withColumn(
    'minutes',
    F.when(
        (F.col('minutes') > 665) | (F.col('minutes').isNull()),
        0
    ).otherwise(F.col("minutes"))
)

#Renaming the columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)
df.display()

#Writing to silver layer

In [0]:
(
df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable('tennislakehouse.silver.matches')
)